In [6]:
%load_ext autoreload
%autoreload 2

%matplotlib inline

import numpy as np
import sys
sys.path.insert(0, "/zhome/71/c/146676/texture_tomography/maptools")
import maptools
import os

import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from multiprocessing import Pool
from datetime import datetime
from tqdm.notebook import tqdm

maptools.job.info()
maptools.plot.dark()
!which python
from scipy.ndimage import generic_filter
from scipy.optimize import curve_fit
from matplotlib.patches import FancyArrowPatch, Circle

from ImageD11 import cImageD11
from ImageD11.sinograms.roi_iradon import run_iradon, run_mlem
from scipy.ndimage import (
    gaussian_filter,
    median_filter,
    maximum_filter,
    binary_fill_holes,
)

from orix import plot
from orix.vector import Vector3d, Miller
from orix.quaternion.symmetry import Oh
from orix.quaternion import Orientation, symmetry
from orix.crystal_map import Phase

import time
import numba
import ImageD11.columnfile
import ImageD11.sinograms.dataset
import ImageD11.sinograms.geometry as geometry
import ImageD11.sinograms.point_by_point
import ImageD11.sinograms.point_by_point as pbp
from ImageD11.nbGui import nb_utils as utils

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
hostname: n-62-11-20
SLURM_JOB_ID: None
SLURM_JOB_NODELIST: None
NPROCS: 24
Memory available: 796.77241344 GB
~/miniconda3/envs/textom/bin/python


In [9]:
savepath = os.path.join(
    maptools.paths.PROCESS, "al1050_15pct_center_slice", "reconstructions"
)
filetag = "medium_pks_18M_2025_1213_095603"

npz_path = os.path.join(savepath, filetag + ".npz")

with np.load(npz_path, allow_pickle=True) as z:
    data = {
        k: z[k]
        for k in z.files
        if k != "grains_raw"
    }

In [10]:
for k in data:
    print(k)


filter_index
rgb_map_filtered
ubi_map_filtered
nuniq_map_filtered
strain_map_filtered
g_map_raw
rgb_map_raw
uniqueness_map_raw
strain_map_raw
orientation_map_filtered
kam_map_filtered
sample_mask
reconstruction_mlem


In [11]:
ubi = data["g_map_raw"]
nuniq = data["uniqueness_map_raw"]
rgb = data["rgb_map_raw"]
mask = data["sample_mask"]

orix_field = np.empty((ubi.shape[0], ubi.shape[1], 5), dtype=object)
nuniq_field = np.empty((ubi.shape[0], ubi.shape[1], 5), dtype=np.int32)
for i in range(ubi.shape[0]):
    for j in range(ubi.shape[1]):
        if mask[i, j]:
            for k in range(5):
                if nuniq[i, j, k] > 0:
                    g = ImageD11.grain.grain(ubi[i, j, k, :, :])
                    g.ref_unitcell = maptools.constants.UNITCELL
                    orix_field[i, j, k] = g.orix_orien
                    nuniq_field[i, j, k] = nuniq[i, j, k]


In [12]:
scirot_field = np.empty((ubi.shape[0], ubi.shape[1], 5), dtype=object)
numpy_matrix_field = np.zeros((ubi.shape[0], ubi.shape[1], 5, 3, 3), dtype=np.float64)
rgb_field = np.zeros((ubi.shape[0], ubi.shape[1], 5, 3, 3), dtype=np.float32)

from scipy.spatial.transform import Rotation as R

for i in range(ubi.shape[0]):
    for j in range(ubi.shape[1]):
        if mask[i, j]:
            for k in range(5):
                if orix_field[i, j, k] is not None:
                    u = orix_field[i, j, k].to_matrix()
                    scirot_field[i, j, k] = R.from_matrix(u)
                    numpy_matrix_field[i, j, k, :, :] = u[:, :]
                    rgb_field[i, j, k, :, :] = rgb[i, j, k, :, :]

In [13]:
from xfab import symmetry

symrots = symmetry.ROTATIONS[7].copy()


def Umis(umat_1, umat_2, symrots):
    lengths = 0.5 * (symrots * np.dot(umat_1.T, umat_2)).sum(axis=(1, 2)) - 0.5
    print(lengths)
    return np.arccos(lengths.clip(-1, 1)).min()

In [14]:
import numba


@numba.njit
def dist(umat_1, umat_2, symrots):
    M = umat_1.T @ umat_2
    n = symrots.shape[0]
    min_val = 1e9
    for i in range(n):
        s = 0.0
        for j in range(3):
            for k in range(3):
                s += symrots[i, j, k] * M[j, k]
        length = 0.5 * s - 0.5
        if length > 1.0:
            length = 1.0
        elif length < -1.0:
            length = -1.0
        val = np.arccos(length)
        if val < min_val:
            min_val = val
    return min_val


@numba.njit
def distance_matrix(us, symrots):
    n = us.shape[0]
    D = np.zeros((n, n), dtype=np.float64)
    for i in range(n):
        for j in range(i):
            D[i, j] = dist(us[i], us[j], symrots)
    for i in range(n):
        D[i, i] = np.inf
    return D


u1 = numpy_matrix_field[100, 100, 0]
u2 = numpy_matrix_field[100, 100, 4]
Umis(u1, u2, symrots), dist(u1, u2, symrots)

[ 0.37376462 -0.98987054 -0.19008903 -0.42601689 -0.3839857  -0.99990837
 -0.68443414 -0.69945993 -0.32429124 -0.94281104 -0.7659211  -0.14740129
 -0.30185252  0.06828787 -0.60793515 -0.97807553 -0.92503372 -0.93814118
 -0.99392606 -0.9808186   0.91481263  0.04859221 -0.99585285 -0.12963244]


(np.float64(0.4157525700249755), 0.4157525700249755)

In [15]:
ubi[100, 100, :]

array([[[ 2.763637, -2.732673, -1.174354],
        [ 2.539905,  3.001362, -0.983551],
        [ 1.527418, -0.068159,  3.753178]],

       [[ 3.086594, -1.765517, -1.930735],
        [ 1.049891,  3.582846, -1.599376],
        [ 2.407057,  0.738755,  3.183852]],

       [[ 2.828373, -2.803288, -0.786457],
        [ 2.458958,  2.894787, -1.427712],
        [ 1.54387 ,  0.511498,  3.70916 ]],

       [[ 2.785647, -2.847051, -0.819103],
        [ 2.658393,  2.876606, -1.042331],
        [ 1.308494,  0.185526,  3.829061]],

       [[ 2.384721, -2.731788,  1.797718],
        [ 1.606614,  2.921692,  2.304371],
        [-2.863469, -0.643483,  2.809032]],

       [[ 2.470291, -2.604153,  1.875596],
        [ 1.610235,  3.057881,  2.125484],
        [-2.780282, -0.553868,  2.898881]],

       [[ 2.987462, -1.658235, -2.171494],
        [ 0.940876,  3.64392 , -1.496044],
        [ 2.582159,  0.603708,  3.082504]],

       [[ 0.      ,  0.      ,  0.      ],
        [ 0.      ,  0.      ,  0.      

In [16]:
us = numpy_matrix_field.copy().reshape(-1, 3, 3).astype(np.float64)
mm = us.sum(axis=(1, 2))
us = us[mm != 0]

rgbs = rgb_field.copy().reshape(-1, 9).astype(np.float64)
rgbs = rgbs[mm != 0]


In [17]:
nn = nuniq_field.copy().reshape(-1)

In [18]:
nn[0:3]

array([0, 0, 0], dtype=int32)

In [19]:
u1 = us[0]
u2 = us[1]
Umis(u1, u2, symrots), dist(u1, u2, symrots)

[ 0.99742383 -0.99973512  0.0218462  -0.02415749 -0.99886981 -0.9988189
 -0.99768899 -0.99999972  0.04787356 -0.43932617 -0.99875532 -0.51155558
 -0.04926864 -0.53651662 -0.99984961 -0.51260162  0.04678988 -0.48961197
 -0.99983634 -0.46343449 -0.04823586 -0.48854155 -0.99871768 -0.55841199]


(np.float64(0.0717952645554948), 0.0717952645554948)

In [20]:
distance_matrix(us[0:5], symrots)

array([[       inf, 0.        , 0.        , 0.        , 0.        ],
       [0.07179526,        inf, 0.        , 0.        , 0.        ],
       [0.07768008, 0.00760103,        inf, 0.        , 0.        ],
       [0.02795751, 0.08569986, 0.08985759,        inf, 0.        ],
       [0.05953495, 0.10167243, 0.10745682, 0.0717756 ,        inf]])

In [21]:
us = numpy_matrix_field.copy().reshape(-1, 3, 3)
mm = us.sum(axis=(1, 2))
us = us[mm != 0]

rgbs = rgb_field.copy().reshape(-1, 9)
rgbs = rgbs[mm != 0]


In [22]:
import numpy as np
from scipy.spatial import cKDTree


def prune_by_rgb_and_dist(us, rgbs, symrots, rgb_threshold, angle_threshold):
    n = us.shape[0]
    tree = cKDTree(rgbs)
    keep = np.ones(n, dtype=bool)

    scores = np.zeros((n,), dtype=np.float64)

    for i in range(n):
        if not keep[i]:
            continue
        idx = tree.query_ball_point(rgbs[i], rgb_threshold)
        for j in idx:
            if j <= i or not keep[j]:
                continue
            D = dist(us[i], us[j], symrots)
            if D < angle_threshold:
                keep[j] = False
                scores[j] = D
    return keep, scores

In [23]:
keep, scores = prune_by_rgb_and_dist(us, rgbs, symrots, 0.6, 1e-8)

In [24]:
np.sum(keep)

np.int64(40078)

In [26]:
print(len(keep))

40078
